# Backtest Framework Comparsion

This notebook compares the same 50/200 SMA crossover portfolio across data vendors and backtesting frameworks. The point is not to make SMA crossover a platform abstraction; it is a compact test case for whether Quant Orchestrator can hold the strategy constant while changing data providers and execution engines.

Data and SMA features come from Quant Warehouse. The backtesting engines receive prepared in-memory frames and do not compute their own indicators.

In [1]:
from __future__ import annotations

from time import perf_counter
import warnings

warnings.filterwarnings("ignore", message="Jupyter Notebook detected.*", category=UserWarning)
warnings.filterwarnings("ignore", message="DataFrameGroupBy.apply operated on the grouping columns.*", category=FutureWarning)
warnings.filterwarnings("ignore", message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*", category=FutureWarning)

import pandas as pd
from IPython.display import Markdown, display


from quant_orchestrator.backtests.research import (
    FrameworkRun,
    _coerce_framework_run,
    _framework_comparison_rows,
    _run_nautilus_isolated,
    build_sma_frame,
    run_backtesting_py,
    run_zipline,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    combine_equity_curves,
    equal_notional_capital,
    load_price_frame,
)
from quant_orchestrator.strategy import summarize_backtest

## Setup

The experiment is a single MAG7 portfolio, not seven independent headline backtests. Each symbol gets an equal capital sleeve and the portfolio equity curve is the sum of those sleeves. We then compare the portfolio-level result for each provider/framework pair.

In [2]:
SYMBOLS = MAG7_SYMBOLS
PROVIDERS = ("yfinance", "fmp")
FRAMEWORKS = ("backtesting.py", "zipline", "nautilus")
START = "2020-01-01"
END = None
FAST_WINDOW = 50
SLOW_WINDOW = 200
CAPITAL_BASE = 100_000.0

CONFIG = pd.DataFrame(
    [
        {
            "symbols": ", ".join(SYMBOLS),
            "providers": ", ".join(PROVIDERS),
            "frameworks": ", ".join(FRAMEWORKS),
            "start": START,
            "end": END or "latest available",
            "fast_window": FAST_WINDOW,
            "slow_window": SLOW_WINDOW,
            "capital_base": CAPITAL_BASE,
            "capital_per_symbol": equal_notional_capital(CAPITAL_BASE, len(SYMBOLS)),
        }
    ]
)
display(CONFIG.T.rename(columns={0: "value"}))

,value
symbols,"AAPL, AMZN, GOOGL, META, MSFT, NVDA, TSLA"
providers,"yfinance, fmp"
frameworks,"backtesting.py, zipline, nautilus"
start,2020-01-01
end,latest available
fast_window,50
slow_window,200
capital_base,100000.0
capital_per_symbol,14285.714286


In [3]:
def load_provider_prices(provider: str) -> dict[str, pd.DataFrame]:
    return {
        symbol: load_price_frame(symbol, provider=provider, start=START, end=END)
        for symbol in SYMBOLS
    }


def run_symbol_sleeve(
    framework: str,
    *,
    symbol: str,
    prices: pd.DataFrame,
    capital_base: float,
) -> FrameworkRun:
    if framework == "nautilus":
        return _run_nautilus_isolated(
            prices,
            symbol=symbol,
            fast_window=FAST_WINDOW,
            slow_window=SLOW_WINDOW,
            capital_base=capital_base,
        )

    signal_frame = build_sma_frame(prices, fast_window=FAST_WINDOW, slow_window=SLOW_WINDOW)
    if framework == "backtesting.py":
        return _coerce_framework_run(run_backtesting_py(signal_frame, symbol=symbol, capital_base=capital_base))
    if framework == "zipline":
        return _coerce_framework_run(run_zipline(signal_frame, symbol=symbol, capital_base=capital_base))
    raise ValueError(f"Unsupported framework: {framework}")


def run_mag7_portfolio(
    *,
    provider: str,
    framework: str,
    price_frames: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, FrameworkRun]]:
    capital_per_symbol = equal_notional_capital(CAPITAL_BASE, len(price_frames))
    started = perf_counter()
    sleeves = []
    runs = {}
    symbol_rows = []
    trades = 0
    native_elapsed_seconds = 0.0

    for symbol, prices in price_frames.items():
        run = run_symbol_sleeve(
            framework,
            symbol=symbol,
            prices=prices,
            capital_base=capital_per_symbol,
        )
        runs[symbol] = run
        sleeves.append(run.equity.rename(symbol))
        row = run.summary.copy()
        row["provider"] = provider
        row["framework"] = framework
        symbol_rows.append(row)
        trades += int(run.summary["trades"].iloc[0])
        native_elapsed_seconds += float(run.summary["elapsed_seconds"].iloc[0])

    portfolio_equity = combine_equity_curves(sleeves)
    wall_clock_seconds = perf_counter() - started
    portfolio_summary = summarize_backtest(
        framework=framework,
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=wall_clock_seconds,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary["provider"] = provider
    portfolio_summary["fast_window"] = FAST_WINDOW
    portfolio_summary["slow_window"] = SLOW_WINDOW
    portfolio_summary["capital_base"] = CAPITAL_BASE
    portfolio_summary["capital_per_symbol"] = capital_per_symbol
    portfolio_summary["native_elapsed_seconds"] = round(native_elapsed_seconds, 4)
    portfolio_summary["wall_clock_seconds"] = round(wall_clock_seconds, 4)
    portfolio_summary["performance_score"] = (
        portfolio_summary["total_return"] + portfolio_summary["max_drawdown"]
    )
    return portfolio_summary, pd.concat(symbol_rows, ignore_index=True), runs

## Run Jobs

Each row below is a Quant Orchestrator-style job: one data provider, one backtesting framework, one equal-notional MAG7 portfolio strategy.

In [4]:
price_frames_by_provider = {provider: load_provider_prices(provider) for provider in PROVIDERS}

portfolio_rows = []
symbol_rows = []
runs: dict[str, dict[str, dict[str, FrameworkRun]]] = {}

for provider in PROVIDERS:
    runs[provider] = {}
    for framework in FRAMEWORKS:
        portfolio_summary, symbol_summary, framework_runs = run_mag7_portfolio(
            provider=provider,
            framework=framework,
            price_frames=price_frames_by_provider[provider],
        )
        portfolio_rows.append(portfolio_summary)
        symbol_rows.append(symbol_summary)
        runs[provider][framework] = framework_runs

comparison = (
    pd.concat(portfolio_rows, ignore_index=True)
    .sort_values(["provider", "framework"])
    .reset_index(drop=True)
)
symbol_detail = pd.concat(symbol_rows, ignore_index=True).sort_values(["provider", "framework", "symbol"]).reset_index(drop=True)
comparison

Loading BokehJS ...

,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second,provider,fast_window,slow_window,capital_base,capital_per_symbol,native_elapsed_seconds,wall_clock_seconds,performance_score
0,backtesting.py,MAG7,2020-01-02,2026-06-24,1627,23,187883.19,0.8788,-0.1454,0.0076,0.4597,3538.92,fmp,50,200,100000.0,14285.714286,0.1937,0.4597,0.7334
1,nautilus,MAG7,2020-01-02,2026-06-24,1627,45,240630.39,1.4063,-0.3076,0.0129,20.8525,78.02,fmp,50,200,100000.0,14285.714286,0.3031,20.8525,1.0987
2,zipline,MAG7,2020-01-02,2026-06-24,1627,60,211225.83,1.1123,-0.2321,0.0124,2.7138,599.53,fmp,50,200,100000.0,14285.714286,2.4786,2.7138,0.8802
3,backtesting.py,MAG7,2020-01-02,2026-06-23,1626,21,131386.57,0.3139,-0.1687,0.0057,0.5095,3191.52,yfinance,50,200,100000.0,14285.714286,0.1430,0.5095,0.1452
4,nautilus,MAG7,2020-01-02,2026-06-23,1626,39,142129.96,0.4213,-0.1681,0.0069,20.6970,78.56,yfinance,50,200,100000.0,14285.714286,0.2275,20.6970,0.2532
5,zipline,MAG7,2020-01-02,2026-06-23,1626,44,143119.15,0.4312,-0.1702,0.0068,2.1800,745.88,yfinance,50,200,100000.0,14285.714286,1.5143,2.1800,0.2610


## Portfolio Comparison

This table is the main result. Returns and drawdowns are portfolio-level values after combining the equal-notional symbol sleeves.

In [5]:
comparison_display = comparison.loc[:, [
    "provider",
    "framework",
    "symbol",
    "start",
    "end",
    "trades",
    "final_equity",
    "total_return",
    "max_drawdown",
    "daily_vol",
    "performance_score",
    "wall_clock_seconds",
    "bars_per_second",
]].copy()
for column in ["total_return", "max_drawdown", "daily_vol", "performance_score"]:
    comparison_display[column] = (comparison_display[column] * 100).round(2).astype(str) + "%"
for column in ["final_equity", "wall_clock_seconds", "bars_per_second"]:
    comparison_display[column] = comparison_display[column].round(2)
display(comparison_display)

,provider,framework,symbol,start,end,trades,final_equity,total_return,max_drawdown,daily_vol,performance_score,wall_clock_seconds,bars_per_second
0,fmp,backtesting.py,MAG7,2020-01-02,2026-06-24,23,187883.19,87.88%,-14.54%,0.76%,73.34%,0.46,3538.92
1,fmp,nautilus,MAG7,2020-01-02,2026-06-24,45,240630.39,140.63%,-30.76%,1.29%,109.87%,20.85,78.02
2,fmp,zipline,MAG7,2020-01-02,2026-06-24,60,211225.83,111.23%,-23.21%,1.24%,88.02%,2.71,599.53
3,yfinance,backtesting.py,MAG7,2020-01-02,2026-06-23,21,131386.57,31.39%,-16.87%,0.57%,14.52%,0.51,3191.52
4,yfinance,nautilus,MAG7,2020-01-02,2026-06-23,39,142129.96,42.13%,-16.81%,0.69%,25.32%,20.70,78.56
5,yfinance,zipline,MAG7,2020-01-02,2026-06-23,44,143119.15,43.12%,-17.02%,0.68%,26.1%,2.18,745.88


## Symbol Detail

The strategy is reported as a MAG7 portfolio, but the symbol-level sleeves are shown here to make the portfolio construction auditable.

In [6]:
symbol_detail_display = symbol_detail.loc[:, [
    "provider",
    "framework",
    "symbol",
    "trades",
    "final_equity",
    "total_return",
    "max_drawdown",
    "elapsed_seconds",
]].copy()
for column in ["total_return", "max_drawdown"]:
    symbol_detail_display[column] = (symbol_detail_display[column] * 100).round(2).astype(str) + "%"
for column in ["final_equity", "elapsed_seconds"]:
    symbol_detail_display[column] = symbol_detail_display[column].round(2)
display(symbol_detail_display)

,provider,framework,symbol,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,fmp,backtesting.py,AAPL,5,17226.20,20.58%,-21.1%,0.03
1,fmp,backtesting.py,AMZN,5,13480.59,-5.64%,-23.49%,0.03
2,fmp,backtesting.py,GOOGL,3,27242.55,90.7%,-15.11%,0.03
3,fmp,backtesting.py,META,3,21362.81,49.54%,-17.73%,0.03
4,fmp,backtesting.py,MSFT,5,17108.09,19.76%,-15.16%,0.03
5,fmp,backtesting.py,NVDA,2,77177.21,440.24%,-27.96%,0.03
6,fmp,backtesting.py,TSLA,0,14285.71,0.0%,0.0%,0.03
7,fmp,nautilus,AAPL,9,17027.75,19.19%,-20.93%,0.05
8,fmp,nautilus,AMZN,9,13313.72,-6.8%,-24.74%,0.05
9,fmp,nautilus,GOOGL,5,27184.83,90.29%,-15.0%,0.05


## Difference Attribution

This is a small two-way decomposition over the 2 providers x 3 frameworks matrix. It is not a statistical test; it is a practical magnitude check for whether the provider choice or framework choice moved a metric more in this fixed strategy example.

In [7]:
factor_report = pd.concat(
    [
        _framework_comparison_rows(comparison, metric="performance_score"),
        _framework_comparison_rows(comparison, metric="total_return"),
        _framework_comparison_rows(comparison, metric="max_drawdown"),
        _framework_comparison_rows(comparison, metric="wall_clock_seconds"),
    ],
    ignore_index=True,
)
factor_display = factor_report.copy()
for column in ["framework_share", "provider_share", "residual_share"]:
    factor_display[column] = (factor_display[column] * 100).round(2).astype(str) + "%"
for column in ["framework_ss", "provider_ss", "residual_ss", "provider_to_framework_ratio"]:
    factor_display[column] = factor_display[column].round(6)
display(factor_display)

,metric,framework_ss,provider_ss,residual_ss,framework_share,provider_share,residual_share,dominant_factor,provider_to_framework_ratio
0,performance_score,0.056228,0.702400,0.019729,7.22%,90.24%,2.53%,provider,12.492061
1,total_return,0.101145,0.829560,0.047057,10.34%,84.84%,4.81%,provider,8.201673
2,max_drawdown,0.006547,0.005287,0.006631,35.46%,28.63%,35.91%,framework,0.807498
3,wall_clock_seconds,500.967263,0.068160,0.087641,99.97%,0.01%,0.02%,framework,0.000136


## Summaries

These summaries average over the other axis. They are useful for quick scanning, but the detailed comparison table above is the source of truth for this run.

In [8]:
framework_summary = comparison.groupby("framework").agg(
    mean_total_return=("total_return", "mean"),
    mean_max_drawdown=("max_drawdown", "mean"),
    mean_performance_score=("performance_score", "mean"),
    mean_wall_clock_seconds=("wall_clock_seconds", "mean"),
    mean_bars_per_second=("bars_per_second", "mean"),
).sort_values("mean_performance_score", ascending=False)
provider_summary = comparison.groupby("provider").agg(
    mean_total_return=("total_return", "mean"),
    mean_max_drawdown=("max_drawdown", "mean"),
    mean_performance_score=("performance_score", "mean"),
    mean_wall_clock_seconds=("wall_clock_seconds", "mean"),
    mean_bars_per_second=("bars_per_second", "mean"),
).sort_values("mean_performance_score", ascending=False)

display(framework_summary)
display(provider_summary)

,mean_total_return,mean_max_drawdown,mean_performance_score,mean_wall_clock_seconds,mean_bars_per_second
framework,,,,,
nautilus,0.91380,-0.23785,0.67595,20.77475,78.290
zipline,0.77175,-0.20115,0.57060,2.44690,672.705
backtesting.py,0.59635,-0.15705,0.43930,0.48460,3365.220


,mean_total_return,mean_max_drawdown,mean_performance_score,mean_wall_clock_seconds,mean_bars_per_second
provider,,,,,
fmp,1.132467,-0.228367,0.9041,8.008667,1405.490000
yfinance,0.388800,-0.169000,0.2198,7.795500,1338.653333


## Written Analysis

The following cell writes the analysis from the outputs above so it changes when the notebook is rerun with a different date range, provider set, framework set, or symbol universe.

In [9]:
def _fmt_pct(value: float) -> str:
    return f"{value * 100:.2f}%"


def _metric_factor(metric: str) -> pd.Series:
    return factor_report.loc[factor_report["metric"] == metric].iloc[0]


best_score = comparison.sort_values("performance_score", ascending=False).iloc[0]
best_return = comparison.sort_values("total_return", ascending=False).iloc[0]
fastest = comparison.sort_values("wall_clock_seconds", ascending=True).iloc[0]
score_factor = _metric_factor("performance_score")
return_factor = _metric_factor("total_return")
speed_factor = _metric_factor("wall_clock_seconds")

analysis = f"""## Analysis From This Run

- Best risk-adjusted simple score: **{best_score['provider']} + {best_score['framework']}** with performance score {_fmt_pct(float(best_score['performance_score']))}, total return {_fmt_pct(float(best_score['total_return']))}, and max drawdown {_fmt_pct(float(best_score['max_drawdown']))}.
- Best raw return: **{best_return['provider']} + {best_return['framework']}** with total return {_fmt_pct(float(best_return['total_return']))}.
- Fastest run: **{fastest['provider']} + {fastest['framework']}** at {float(fastest['wall_clock_seconds']):.2f} wall-clock seconds for the full MAG7 portfolio run.
- For performance score, the larger observed driver was **{score_factor['dominant_factor']}**: framework share {score_factor['framework_share']:.1%}, provider share {score_factor['provider_share']:.1%}, residual share {score_factor['residual_share']:.1%}.
- For raw total return, the larger observed driver was **{return_factor['dominant_factor']}**: framework share {return_factor['framework_share']:.1%}, provider share {return_factor['provider_share']:.1%}, residual share {return_factor['residual_share']:.1%}.
- For speed, the larger observed driver was **{speed_factor['dominant_factor']}**: framework share {speed_factor['framework_share']:.1%}, provider share {speed_factor['provider_share']:.1%}, residual share {speed_factor['residual_share']:.1%}.

Interpretation: this notebook is useful because it separates three concerns that otherwise get mixed together: vendor data differences, framework execution/reporting differences, and portfolio construction. If the dominant factor changes when the strategy or universe changes, Quant Orchestrator should make that visible without rewriting the whole research workflow.
"""
display(Markdown(analysis))

## Analysis From This Run

- Best risk-adjusted simple score: **fmp + nautilus** with performance score 109.87%, total return 140.63%, and max drawdown -30.76%.
- Best raw return: **fmp + nautilus** with total return 140.63%.
- Fastest run: **fmp + backtesting.py** at 0.46 wall-clock seconds for the full MAG7 portfolio run.
- For performance score, the larger observed driver was **provider**: framework share 7.2%, provider share 90.2%, residual share 2.5%.
- For raw total return, the larger observed driver was **provider**: framework share 10.3%, provider share 84.8%, residual share 4.8%.
- For speed, the larger observed driver was **framework**: framework share 100.0%, provider share 0.0%, residual share 0.0%.

Interpretation: this notebook is useful because it separates three concerns that otherwise get mixed together: vendor data differences, framework execution/reporting differences, and portfolio construction. If the dominant factor changes when the strategy or universe changes, Quant Orchestrator should make that visible without rewriting the whole research workflow.
